In [ ]:


# ---- 1) Correlation-based pruning (fast, model-agnostic) ----
corr_thresh = 0.95  # high threshold so we only drop near-duplicates
Xf = X_features.copy()

# Keep only numeric columns for correlation/VIF
num_cols = Xf.select_dtypes(include=[np.number]).columns.tolist()
Xn = Xf[num_cols].copy()

corr = Xn.corr().abs()
upper = corr.where(np.triu(np.ones(corr.shape), k=1).astype(bool))
to_drop_corr = [col for col in upper.columns if any(upper[col] > corr_thresh)]
Xn_pruned = Xn.drop(columns=to_drop_corr)

print(
    f"[Correlation] Dropped {len(to_drop_corr)} near-duplicate features (>{corr_thresh})."
)

# ---- 2) Optional VIF loop (drops features with extreme multicollinearity) ----
try:
    from statsmodels.stats.outliers_influence import variance_inflation_factor

    def compute_vif(df):
        X_arr = df.values.astype(float)
        vifs = [variance_inflation_factor(X_arr, i) for i in range(X_arr.shape[1])]
        return pd.Series(vifs, index=df.columns, name="VIF")

    vif_thresh = 12.0
    Xv = Xn_pruned.copy()
    dropped_vif = []

    while True:
        vifs = compute_vif(
            Xv.fillna(0.0)
        )  # simple fill for stability; models will get proper imputers if needed
        worst = vifs.sort_values(ascending=False).iloc[0]
        worst_feat = vifs.sort_values(ascending=False).index[0]
        if worst <= vif_thresh or Xv.shape[1] <= 2:
            break
        dropped_vif.append((worst_feat, float(worst)))
        Xv = Xv.drop(columns=[worst_feat])

    X_final = Xv.copy()
    print(f"[VIF] Dropped {len(dropped_vif)} features with VIF>{vif_thresh}.")
    if dropped_vif[:5]:
        print("  Top drops:", dropped_vif[:5])

except Exception as e:
    # If statsmodels not present, proceed with correlation-pruned set
    print(f"[VIF] Skipped VIF step ({e}). Using correlation-pruned set only.")
    X_final = Xn_pruned

# Keep alignment to rows in X / y
final_feature_cols = X_final.columns.tolist()
X_model = X[["m_ref"]].join(X_final)  # keep m_ref for temporal splits
y_model = y.copy()

print(
    {
        "final_n_features": len(final_feature_cols),
        "sample_features": final_feature_cols[:10],
    }
)


In [ ]:
import numpy as np
import pandas as pd
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    precision_recall_curve,
    average_precision_score,
    f1_score,
    brier_score_loss,
    roc_auc_score,
)
from sklearn.calibration import CalibratedClassifierCV
from sklearn.pipeline import Pipeline

# ---- 1) Time-aware split (train / val / test by m_ref chronology) ----
q_train_end = X_model["m_ref"].quantile(0.60)
q_val_end = X_model["m_ref"].quantile(0.80)

train_idx = X_model["m_ref"] <= q_train_end
val_idx = (X_model["m_ref"] > q_train_end) & (X_model["m_ref"] <= q_val_end)
test_idx = X_model["m_ref"] > q_val_end

X_train = X_model.loc[train_idx, final_feature_cols]
y_train = y_model.loc[train_idx]
X_val = X_model.loc[val_idx, final_feature_cols]
y_val = y_model.loc[val_idx]
X_test = X_model.loc[test_idx, final_feature_cols]
y_test = y_model.loc[test_idx]

print(
    {
        "train_size": X_train.shape,
        "val_size": X_val.shape,
        "test_size": X_test.shape,
    }
)

# ---- 2) Two models: calibrated Logistic (baseline) and calibrated RandomForest (nonlinear) ----
# Logistic pipeline (scaling + class_weight)
logit_base = Pipeline(
    [
        ("scaler", StandardScaler(with_mean=True, with_std=True)),
        (
            "clf",
            LogisticRegression(
                max_iter=2000, class_weight="balanced", solver="saga", n_jobs=None
            ),
        ),
    ]
)
logit_base.fit(X_train, y_train)

# Calibrate on validation slice (isotonic). cv='prefit' means we pass a fitted estimator.
logit_cal = CalibratedClassifierCV(logit_base, cv="prefit", method="isotonic")
logit_cal.fit(X_val, y_val)

# RandomForest baseline + calibration
rf_base = RandomForestClassifier(
    n_estimators=400,
    max_depth=None,
    min_samples_leaf=2,
    class_weight="balanced",
    n_jobs=-1,
    random_state=42,
)
rf_base.fit(X_train, y_train)
rf_cal = CalibratedClassifierCV(rf_base, cv="prefit", method="isotonic")
rf_cal.fit(X_val, y_val)


# ---- 3) Evaluate on TEST (PR-AUC, F1@best_threshold, Brier, ROC-AUC), and choose cost-optimal threshold ----
def evaluate_and_choose(model, X_te, y_te, offer_cost=50.0, churn_loss=500.0):
    proba = model.predict_proba(X_te)[:, 1]
    pr_auc = average_precision_score(y_te, proba)
    roc = roc_auc_score(y_te, proba)
    brier = brier_score_loss(y_te, proba)

    # Threshold sweep for F1 and Business Utility
    thresholds = np.linspace(0.05, 0.95, 91)
    best = {"f1": -1, "thr_f1": 0.5, "util": -1e18, "thr_util": 0.5}
    for thr in thresholds:
        pred = (proba >= thr).astype(int)
        f1 = f1_score(y_te, pred, average="macro")
        # Business utility: if we target (pred==1), we pay offer_cost; if y==1 and pred==0, we incur churn_loss
        tp = ((pred == 1) & (y_te == 1)).sum()
        fp = ((pred == 1) & (y_te == 0)).sum()
        fn = ((pred == 0) & (y_te == 1)).sum()
        # Assume we only pay offer cost on targeted customers (pred==1); benefit is avoiding churn loss on TPs
        util = (tp * churn_loss) - ((tp + fp) * offer_cost) - (fn * churn_loss)
        if f1 > best["f1"]:
            best["f1"], best["thr_f1"] = f1, thr
        if util > best["util"]:
            best["util"], best["thr_util"] = util, thr
    return {
        "pr_auc": pr_auc,
        "roc_auc": roc,
        "brier": brier,
        "best_f1_macro": best["f1"],
        "thr_f1_macro": best["thr_f1"],
        "best_expected_utility": best["util"],
        "thr_expected_utility": best["thr_util"],
    }


logit_metrics = evaluate_and_choose(logit_cal, X_test, y_test)
rf_metrics = evaluate_and_choose(rf_cal, X_test, y_test)

print("\n[Calibrated Logistic] Test metrics:", logit_metrics)
print("[Calibrated RandomForest] Test metrics:", rf_metrics)


# Select the winner by PR-AUC (primary) then expected utility
def pick_winner(m1, m2, name1="Logit", name2="RF"):
    key = "pr_auc"
    if m1[key] > m2[key] + 1e-6:
        return name1, m1
    elif m2[key] > m1[key] + 1e-6:
        return name2, m2
    # tie-breaker: expected utility
    return (
        (name1, m1)
        if m1["best_expected_utility"] >= m2["best_expected_utility"]
        else (name2, m2)
    )


winner_name, winner_metrics = pick_winner(logit_metrics, rf_metrics)
print(f"\n>>> Winner: {winner_name}")
print(">>> Metrics:", winner_metrics)
